# Setup Serial Communication
Initialize serial connection to the STM32 via USB (COM port). Make sure to replace `COMx` with your actual STM32 port (e.g., `COM3`, `/dev/ttyUSB0`).

In [11]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'test'))

from ax12_protocol import AX12Controller

# Replace 'COM10' with your actual port
try:
    controller = AX12Controller('COM10', baudrate=115200, timeout=0.5, debug=True)
    print("✓ Connected to STM32 AX-12 Bridge")
except Exception as e:
    print(f"✗ Error: {e}")
    controller = None

Connected to STM32


# Function to Send Commands
The STM32 is configured to read simple ASCII commands in the format `ID,POSITION\n`. The STM32 parses this and builds the Dynamixel packet.

In [12]:
def move_ax12(motor_id, position):
    """
    Sends a movement command to an AX-12 servo.
    motor_id: The ID of the AX-12 servo (e.g. 6 or 14)
    position: The target position (0 to 1023)
    """
    if controller:
        print(f"Moving servo {motor_id} to position {position}")
        controller.set_position(motor_id, position)
    else:
        print("Controller not connected")

# Send Commands to Motor ID 6 and ID 14
Test the servos by moving them to specific positions. AX-12 positions range from 0 (0 degrees) to 1023 (300 degrees).

In [15]:
# First, enable torque on both servos
if controller:
    print("Enabling torque...")
    controller.set_torque_enable(6, True)
    controller.set_torque_enable(14, True)
    print()

# Move Motor ID 6 to position 512 (center approx)
print("Moving Motor 6...")
move_ax12(6, 512)
print()

# Move Motor ID 14 to position 512
print("Moving Motor 14...")
move_ax12(14, 512)
print()

# Test sync write (move both at once)
if controller:
    print("Moving both servos together using SYNC_WRITE...")
    controller.sync_positions(6, 256, 14, 768)
    print("✓ Sync command sent")

Moving Motor 6...
Moving ID 6 to position 512
TX: 0xFF 0xFF 0x6 0x5 0x3 0x1E 0x0 0x2 0xD1
RX: 0xFF 0xFF 0xE 0x5 0x3 0x1E -> ERROR: 0b11

Moving Motor 14...
Moving ID 14 to position 512
TX: 0xFF 0xFF 0xE 0x5 0x3 0x1E 0x0 0x2 0xC9
RX: 0xFF 0xFF 0x6 0x5 0x3 0x1E -> ERROR: 0b11


# Close the Connection

In [16]:
if controller:
    controller.close()
    print("Serial connection closed.")

Serial connection closed.
